# TVLA on single-word Barrett reduction

This notebook checks if HQC's Barrett reduction leaks information through the power it uses.

- The board runs the reduction many times, and the Husky records the power each time.
- In the fixed group, the input is always `FIXED_X`. In the random group, the input is a new random 24-bit number each time.
- If the power differs between the two groups, the code leaks.
- The test is repeated on two halves of the data. A sample leaks only if both halves show it.

In [1]:
import os
import time
from datetime import datetime

import numpy as np
import chipwhisperer as cw
import holoviews as hv
from scipy.stats import ttest_ind
from tqdm.auto import tqdm

N = 17669           # the modulus, must be the same as PARAM_N in simpleserial-barrett.c
FIXED_X = 1212218   # the input used for every fixed-group run
N_WARMUP = 1000     # runs done first to warm up the board, their traces are not kept
N_TRACES = 10000    # test runs, fixed and random together
SAMPLES = 1000      # power samples recorded in each run
SEED = 1            # the same seed gives the same inputs every time
THRESHOLD = 4.5     # the |t| limit, above it in both tests means a leak
ZOOM_START = None   # first clock cycle in the cycle plot, None means the start of the recording
ZOOM_END = None     # last clock cycle in the cycle plot, None means the end of the recording

HEX = "simpleserial-barrett/simpleserial-barrett-CWHUSKY.hex"
ELF = "simpleserial-barrett/simpleserial-barrett-CWHUSKY.elf"
RUN = "tvla_" + datetime.now().strftime("%Y%m%d_%H%M%S")
FIGURES = "figures"  # folder for the PNG copies of the plots

# Bokeh shows the interactive plots, matplotlib draws the PNG copies
hv.extension("bokeh", "matplotlib", logo=False)

## Build

Compiles the C code for the target board.

In [2]:
%%bash
make -C simpleserial-barrett

make: Entering directory '/Users/shubham/Desktop/ResProj/sca-barrett-reduction/simpleserial-barrett'
SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
.
Welcome to another exciting ChipWhisperer target build!!
arm-none-eabi-gcc (Arm GNU Toolchain 15.3.Rel2 (Build arm-15.181)) 15.3.1 20260627
Copyright (C) 2025 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
Compiling:
-en     simpleserial-barrett.c ...
-e Done!
.
Compiling:
-en     ../../chipwhisperer/firmware/mcu/simpleserial/simpleserial.c ...
-e Done!
.
Compiling:
-en     ../../chipwhisperer/firmware/mcu/hal/hal.c ...
-e Done!
.
Compiling:
-en     ../../chipwhisperer/firmware/mcu/hal//sam4s/startup_sam4s.c ...
-e Done!
.
Compiling:
-en     ../../chipwhisperer/firmware/mcu/hal//sam4s/sam4s_hal.c ...
-e Done!
.
Compiling:
-en     ../../chipwhisperer/firmware/mcu/hal//sam4s/ua

## Setup

Connects to the Husky, loads the firmware onto the board, and records one test trace to find where the barrett reduction sits in the recording.

- **Trigger high**: the sample where the board raises the trigger, before the reduction. The rest of `trigger_high` still runs after this point. It equals `scope.adc.presamples`, the number of samples recorded before the trigger. It is 0 unless pre-samples are set.
- **Trigger low**: the sample where the board lowers the trigger, just after the reduction. It equals trigger high plus `scope.adc.trig_count`, the number of samples the trigger stayed high.
- **`SAMPLES_PER_CYCLE`**: how many power samples the Husky takes during one clock cycle of the board. It is used later to turn samples into clock cycles.

The recording must reach trigger low, so the cell stops with an error if `SAMPLES` is too small.

In [3]:
# Connect to the Husky and the target board
scope = cw.scope()
target = cw.target(scope)
scope.default_setup()

# Load the firmware onto the board
cw.program_target(scope, cw.programmers.SAM4SProgrammer, HEX)

# Record SAMPLES samples per run, and find how many samples fit in one board clock cycle
scope.adc.samples = SAMPLES
SAMPLES_PER_CYCLE = round(scope.clock.adc_freq / scope.clock.clkgen_freq)

# Sends x to the target board, runs one reduction, and returns the recorded power.
# Stops with an error if nothing is recorded, or the board gives no answer or a
# wrong answer, so a trace is never saved with the wrong input.
def capture(x):
    scope.arm()
    target.simpleserial_write("b", bytearray(x.to_bytes(3, "little")))

    timed_out = scope.capture()
    if timed_out:
        raise RuntimeError(f"x={x}: scope timed out, no trace recorded")

    reply = target.simpleserial_read("r", 4)
    if reply is None:
        raise RuntimeError(f"x={x}: no reply from the board")

    result = int.from_bytes(reply, "little")
    expected = x % N
    if result != expected:
        raise RuntimeError(f"x={x}: board sent {result}, expected {expected}")

    return scope.get_last_trace()


# Run once to find where the trigger goes high and low in the recording
capture(FIXED_X)
TRIGGER_HIGH = scope.adc.presamples
TRIGGER_LOW = TRIGGER_HIGH + scope.adc.trig_count
trigger_samples = TRIGGER_LOW - TRIGGER_HIGH
print(f"trigger high at sample {TRIGGER_HIGH}")
print(f"trigger low at sample {TRIGGER_LOW}")
print(f"trigger high for {trigger_samples} samples = {trigger_samples // SAMPLES_PER_CYCLE} cycles")

# The recording must reach trigger low, or the end of the reduction is cut off
if TRIGGER_LOW > SAMPLES:
    raise ValueError(f"set SAMPLES to at least {TRIGGER_LOW}")

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_

## Inputs

Makes the inputs. Trace `i` will belong to input `i`. Group 0 is fixed and group 1 is random. The two groups are mixed in a random order, so slow changes in the board affect both groups the same way.

In [4]:
rng = np.random.default_rng(SEED)

# Pick the group of each run at random (0 = fixed, 1 = random),
# then use FIXED_X for group 0 and a random 24-bit number for group 1
groups = rng.integers(0, 2, N_TRACES)
inputs = np.where(groups == 0, FIXED_X, rng.integers(0, 2**24, N_TRACES))

## Warm-up

Runs the reduction `N_WARMUP` times so the board reaches a steady temperature. These traces are thrown away. A flat line in the plot means the board is warm.

In [5]:
# Saves a PNG copy of a plot as FIGURES/<name>.png, so it can be shown on GitHub.
# Returns the plot unchanged, so the notebook still shows the interactive version.
def save_png(figure, name):
    os.makedirs(FIGURES, exist_ok=True)
    wide = figure.opts(hv.opts.Curve(linewidth=0.8, aspect=2, fig_inches=10, backend="matplotlib"),
                       hv.opts.Overlay(aspect=2, fig_inches=10, backend="matplotlib"), clone=True)
    hv.save(wide, os.path.join(FIGURES, name + ".png"), backend="matplotlib", dpi=100)
    return figure


warmup_inputs = rng.integers(0, 2**24, N_WARMUP)
mean_power = np.zeros(N_WARMUP)

# Run the reduction on each warm-up input and keep only the average power of the trace
start = time.time()
for i, x in enumerate(tqdm(warmup_inputs, desc="warm-up")):
    trace = capture(int(x))
    mean_power[i] = trace.mean()
print(f"warm-up: {time.time() - start:.1f} s")

# One point per run. The line should settle and stay flat once the board is warm.
save_png(cw.plot(mean_power, "warm-up trace", "mean power"), "warmup")

warm-up:   0%|          | 0/1000 [00:00<?, ?it/s]

warm-up: 22.2 s


:Curve   [warm-up trace]   (mean power)

## Capture

Records one trace for every input, in the same order as the inputs, and saves them.

In [6]:
traces = np.zeros((N_TRACES, SAMPLES), dtype=np.float32)

start = time.time()
for i, x in enumerate(tqdm(inputs, desc="capture")):
    traces[i] = capture(int(x))
minutes = (time.time() - start) / 60

n_fixed = np.sum(groups == 0)
n_random = np.sum(groups == 1)
print(f"capture: {minutes:.1f} min, {n_fixed} fixed traces, {n_random} random traces")

# Save each trace next to its input and group, in the same order
np.savez(RUN + "_traces.npz", traces=traces, inputs=inputs, groups=groups)
print(f"traces saved to {RUN}_traces.npz")

# Save the same data as a table: one row per trace with its input, group and samples
table = np.column_stack([inputs, groups, traces])
header = ",".join(["input", "group"] + [f"sample_{i}" for i in range(SAMPLES)])
np.savetxt(RUN + "_traces.csv", table, delimiter=",", header=header, comments="", fmt=["%d", "%d"] + ["%g"] * SAMPLES)
print(f"table saved to {RUN}_traces.csv")

capture:   0%|          | 0/10000 [00:00<?, ?it/s]

capture: 3.7 min, 4997 fixed traces, 5003 random traces
traces saved to tvla_20261115_182810_traces.npz
table saved to tvla_20261115_182810_traces.csv


## Saved files

Each notebook run saves the same data in two files. `RUN` is the date and time the notebook started, and the Capture cell prints both file names.

- `<RUN>_traces.npz` is for Python and loads fast.
- `<RUN>_traces.csv` is a table you can open in a spreadsheet, a text editor or pandas.

The plots are also saved as PNG images in the `figures` folder, so they can be shown on GitHub: `warmup.png`, `means.png`, `difference.png`, `welch.png`, `halves.png` and `cycles.png`. Each run replaces them with its own plots.

The `.npz` file holds three arrays:

| Key | Content |
| --- | --- |
| `traces` | one row per trace, with `SAMPLES` power readings each |
| `inputs` | the input `x` of each trace, in capture order |
| `groups` | the group of each trace: 0 is fixed, 1 is random |

Row `i` of `traces` is the trace recorded for `inputs[i]`, which is in group `groups[i]`.

The `.csv` file has one row per trace, in capture order:

| Column | Content |
| --- | --- |
| `input` | the input `x` of the trace |
| `group` | the group of the trace: 0 is fixed, 1 is random |
| `sample_0`, `sample_1`, ... | the power readings of the trace, one column per sample |

To read the files later, open a notebook or Python in this folder and change the file names to yours:

```python
import numpy as np
import pandas as pd

# The arrays from the .npz file
data = np.load("tvla_20261115_170000_traces.npz")
traces = data["traces"]
inputs = data["inputs"]
groups = data["groups"]

# The first trace with its input and group
print(traces[0], inputs[0], groups[0])

# All traces of the fixed group
fixed_traces = traces[groups == 0]

# The same data as a table from the .csv file
table = pd.read_csv("tvla_20261115_170000_traces.csv")
table.head()
```

## Mean of each group

A single trace is noisy: the power changes a little on every run, even when the input is the same. Averaging many traces removes most of this noise and leaves the power the code really uses at each sample.

The plot shows two averages:

- **fixed**: the average of all traces where the input was `FIXED_X`.
- **random**: the average of all traces with a random input.

**How to read it.** The two lines should have the same shape, because the board runs the same instructions for every input. The shape follows the program: the end of `trigger_high`, then the reduction, then the reply after the trigger goes low.

- **Lines on top of each other**: the input does not change the power at that sample.
- **Lines apart**: the fixed input uses a different amount of power than random inputs there, so the power depends on the data.

In [7]:
# Split the traces into the two groups
fixed_traces = traces[groups == 0]
random_traces = traces[groups == 1]

# Average each group, sample by sample
mean_fixed = fixed_traces.mean(axis=0)
mean_random = random_traces.mean(axis=0)

plot_random = cw.plot(mean_random, "sample", "mean power", label="random")
plot_fixed = cw.plot(mean_fixed, "sample", "mean power", label="fixed")
save_png(plot_random * plot_fixed, "means")

:Overlay
   .Curve.Random :Curve   [sample]   (mean power)
   .Curve.Fixed  :Curve   [sample]   (mean power)

The lines are drawn on the scale of the whole trace, so a gap of a fraction of a unit cannot be seen, even when it is a real leak. A visible gap is not proof either. Each average still holds some noise, so a small gap can appear by chance, and this plot has no limit that separates a real gap from noise. The next plot shows the gap on its own, and the Welch t-test after it gives that limit.

## Difference of the means

This plot is the fixed average minus the random average, sample by sample. It makes the small gaps between the two lines of the previous plot easy to see.

**How to read it.**

- **Close to 0**: at this sample, the power does not depend on the input.
- **A peak or a dip**: at this sample, the fixed input uses a different amount of power than random inputs do on average. The power depends on the data, so the code may leak here.

In [8]:
difference = mean_fixed - mean_random
save_png(cw.plot(difference, "sample", "fixed - random"), "difference")

:Curve   [sample]   (fixed - random)

The difference alone cannot prove a leak. Some samples are noisier than others, and with a limited number of traces a small gap can appear by chance. The difference also has no fixed scale to compare against a limit. The t-test in the next step solves both problems.

## Welch t-test

The t-test checks at each sample whether the gap between the two averages is larger than noise alone would give. It divides the difference of the means by how much that difference can move by chance:

```
t = (mean_fixed - mean_random) / sqrt(var_fixed / n_fixed + var_random / n_random)
```

- `var` is the spread (variance) of a group at that sample.
- `n` is the number of traces in the group.
- Welch's version does not assume the two groups have the same spread. This matters here: the fixed group always uses the same input, so its spread is only noise, while the random group also changes with the input.

**How to read it.** The two flat lines are +4.5 and -4.5.

- **|t| below 4.5**: the gap could be noise. The test shows no leak at this sample.
- **|t| above 4.5**: if there were only noise, this would happen about once in 150,000 samples. The power at this sample very likely depends on the input.
- **The sign** only says which group gave the higher average reading. A positive t means the fixed group's average reading was higher.
- **A larger |t|** means a clearer leak, but |t| also grows with the number of traces. Compare |t| only between runs with the same number of traces.

In [9]:
# Returns the Welch t-value at each sample between the traces in a and in b.
def welch_t(a, b):
    return ttest_ind(a, b, axis=0, equal_var=False).statistic


# Plots each t-value curve in curves ({label: t}) against x,
# with lines at +THRESHOLD and -THRESHOLD.
def plot_t(x, curves, xlabel):
    bound = np.full(len(x), THRESHOLD)
    figure = cw.plot((x, bound), xlabel, "t") * cw.plot((x, -bound), xlabel, "t")
    for label, t in curves.items():
        figure *= cw.plot((x, t), xlabel, "t", label=label)
    return figure


# The sample numbers, used as the x axis of the t-test plots
sample_index = np.arange(SAMPLES)

t_all = welch_t(fixed_traces, random_traces)
save_png(plot_t(sample_index, {"all traces": t_all}, "sample"), "welch")

:Overlay
   .Curve.I          :Curve   [sample]   (t)
   .Curve.II         :Curve   [sample]   (t)
   .Curve.All_traces :Curve   [sample]   (t)

One t-test is not enough to call a leak. Each trace has `SAMPLES` samples, and each one is a new chance for noise to cross 4.5. With the default of 1000 samples, noise alone can cross it somewhere in up to about 1 test in 150. The next section repeats the test on two halves of the data, and a real leak must show up in both.

## Welch t-test on two halves

This is the repeat check from the ChipWhisperer AES tutorial. The traces are cut in two:

- **half 1**: the first half of the fixed traces against the first half of the random traces.
- **half 2**: the second half of each group.

**How to read it.** A sample leaks when both halves give |t| above 4.5 with the same sign. A real leak comes from the code, so it appears at the same sample in both halves and points the same way. A noise spike usually appears in only one half.

In [10]:
# Cut each group in two and test each half
half_fixed = len(fixed_traces) // 2
half_random = len(random_traces) // 2
t_half1 = welch_t(fixed_traces[:half_fixed], random_traces[:half_random])
t_half2 = welch_t(fixed_traces[half_fixed:], random_traces[half_random:])

# A sample leaks when |t| is above the threshold in both halves, with the same sign
both_above = (np.abs(t_half1) > THRESHOLD) & (np.abs(t_half2) > THRESHOLD)
same_sign = np.sign(t_half1) == np.sign(t_half2)
leak = both_above & same_sign

print(f"leaking samples: {np.sum(leak)}, "
      f"max |t|: {np.max(np.abs(t_half1)):.1f} and {np.max(np.abs(t_half2)):.1f}")
save_png(plot_t(sample_index, {"half 1": t_half1, "half 2": t_half2}, "sample"), "halves")

leaking samples: 71, max |t|: 77.9 and 78.4


:Overlay
   .Curve.I      :Curve   [sample]   (t)
   .Curve.II     :Curve   [sample]   (t)
   .Curve.Half_1 :Curve   [sample]   (t)
   .Curve.Half_2 :Curve   [sample]   (t)

This test has limits. The two halves are one dataset split after recording, not two separate recordings. Each half also has only half of the traces, so its |t| is about 1.4 times (the square root of 2) smaller than for all the traces. A weak leak can stay below 4.5 in the halves even when the t-test on all traces shows it.

## Where it leaks

Shows the leaking samples as clock cycles of the board, using the two-halves result. The board's clock drives the processor, so instruction timing is counted in clock cycles. This makes cycles easier than samples to compare with the instructions.

- **Cycle 0** is trigger low, a few cycles after the reduction ends.
- **Negative cycles** come before trigger low. They hold the whole trigger window: first the end of `trigger_high` (setup code that does not touch `x`), then the reduction, then the start of `trigger_low`.
- **Positive cycles** come after trigger low. The board packs and sends the reply there. The reply is `x mod N`, so it can differ between groups too, but it is not part of the reduction.

**The cycle plot.** With `ZOOM_START = None` and `ZOOM_END = None` in the settings cell, the plot shows every cycle of the recording, from the first sample to the last. To zoom in, set them to cycle numbers. For example, -60 and 20 show the reduction with a margin on each side, because the reduction is expected in roughly the last 40 to 50 cycles before cycle 0. This is an estimate from the instructions, not a measurement.

In [11]:
# Turn each sample into a clock cycle, counted from trigger low (cycle 0)
cycles = (sample_index - TRIGGER_LOW) / SAMPLES_PER_CYCLE

leak_cycles = np.unique(np.floor(cycles[leak]).astype(int))
print("leaking cycles:", leak_cycles.tolist())

# Plot the cycles from ZOOM_START to ZOOM_END. None means the first or the last cycle.
first = cycles.min() if ZOOM_START is None else ZOOM_START
last = cycles.max() if ZOOM_END is None else ZOOM_END
zoom = (cycles >= first) & (cycles <= last)
save_png(plot_t(cycles[zoom], {"half 1": t_half1[zoom], "half 2": t_half2[zoom]}, "cycles relative to trigger low"), "cycles")

leaking cycles: [-129, -42, -38, -35, -33, -32, -31, -30, -29, -28, -27, -26, -25, -24, -22, -21, -20, -18, -17, -16, -10, 22, 104, 105, 106, 107, 108, 109]


:Overlay
   .Curve.I      :Curve   [cycles relative to trigger low]   (t)
   .Curve.II     :Curve   [cycles relative to trigger low]   (t)
   .Curve.Half_1 :Curve   [cycles relative to trigger low]   (t)
   .Curve.Half_2 :Curve   [cycles relative to trigger low]   (t)

The cycle numbers point to a region of the code, not to one exact instruction. The processor works on more than one instruction at a time, and calls and memory reads take extra cycles, so a leak can show up a few cycles away from the instruction that causes it. Compare the leaking cycles with the instructions printed below.

In [12]:
# Print the board instructions for the 'b' command
!arm-none-eabi-objdump --disassemble=handle_barrett {ELF}


simpleserial-barrett/simpleserial-barrett-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400158 <handle_barrett>:
  400158:	b538      	push	{r3, r4, r5, lr}
  40015a:	7883      	ldrb	r3, [r0, #2]
  40015c:	7842      	ldrb	r2, [r0, #1]
  40015e:	4d16      	ldr	r5, [pc, #88]	@ (4001b8 <handle_barrett+0x60>)
  400160:	041b      	lsls	r3, r3, #16
  400162:	ea43 2302 	orr.w	r3, r3, r2, lsl #8
  400166:	7802      	ldrb	r2, [r0, #0]
  400168:	4313      	orrs	r3, r2
  40016a:	602b      	str	r3, [r5, #0]
  40016c:	4b13      	ldr	r3, [pc, #76]	@ (4001bc <handle_barrett+0x64>)
  40016e:	4604      	mov	r4, r0
  400170:	4798      	blx	r3
  400172:	682a      	ldr	r2, [r5, #0]
  400174:	4b12      	ldr	r3, [pc, #72]	@ (4001c0 <handle_barrett+0x68>)
  400176:	4d13      	ldr	r5, [pc, #76]	@ (4001c4 <handle_barrett+0x6c>)
  400178:	fba3 1302 	umull	r1, r3, r3, r2
  40017c:	f244 5105 	movw	r1, #17669	@ 0x4505
  400180:	fb01 2213 	mls	r2, r1, r3, r2
  400184:	f5a2 438a 	su

## What each instruction does

The explanation of each instruction in the listing above is in [docs/barrett-instructions.md](docs/barrett-instructions.md).

## Disconnect

Releases the Husky so other notebooks and programs can use it.

In [13]:
scope.dis()
target.dis()